In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#biblioteca para fazer o tratamento das palavras com acento
import unicodedata 

* Função para realizar a normalização dos nomes das cidades para caixa alta e sem acento

In [2]:
def normalizar_texto(texto):

    if not isinstance(texto, str):
        return ""
    
    # Separa os acentos das letras
    texto_normalizado = unicodedata.normalize('NFKD', texto)

    # Conversão para ASCII ignorando os acentos
    texto_ascii = texto_normalizado.encode('ASCII', 'ignore').decode('ASCII')

    #Transformação das letras em maiúsculas
    texto = texto_ascii.upper().strip()

    return texto

* Leitura da base de dados MG 2020 a 2025

In [3]:
anos = [2020, 2021, 2022, 2023, 2024, 2025]
lista_dfs = []

for ano in anos:
    # Leitura dos arquivos em excel
    caminho = f'2 - MG/RelatorioEstadoNutricional {ano} MG.xls.xlsx'
    df_ano = pd.read_excel(caminho, header=1)

    # Remove espaços em branco antes e depois do nome de todas as colunas
    df_ano.columns = df_ano.columns.astype(str).str.strip()

    # Acrescenta a coluna do ano da base de dados
    df_ano['ano'] = ano

    # Aplica a normalização no nome dos municípios
    df_ano['municipio_norm'] = df_ano['Município'].apply(normalizar_texto)

    # Altera o tipo de dados das colunas de quantidades para numérico

    colunas_qtd = [ 'Baixo peso','Adequado','Sobrepeso', 'Obesidade Grau I', 'Obesidade Grau II', 'Obesidade Grau III', 'Total']

    for coluna in colunas_qtd:
        df_ano[coluna] = pd.to_numeric(df_ano[coluna], errors='coerce')


    # Tratamento da coluna de dados que possuem o símbolo de porcentagem nas células
    colunas_porcentagem = ['Obesidade Grau I %', 'Obesidade Grau II %', 'Obesidade Grau III %']

    for coluna in colunas_porcentagem:
        # Tipo da coluna é string
        df_ano[coluna] = df_ano[coluna].astype(str)

        # Remoção do símbolo de porcentagem
        df_ano[coluna] = df_ano[coluna].str.replace('%', '', regex=False).str.replace(',', '.', regex=False).str.strip()

        # Conversão para float
        df_ano[coluna] = pd.to_numeric(df_ano[coluna], errors='coerce')

        # Verifica se há alguma célula fora do padrão de 100%
        condicao = (df_ano[coluna] > 0) & (df_ano[coluna] <= 1)

        df_ano.loc[condicao, coluna] = df_ano[coluna] * 100

        df_ano[coluna] = df_ano[coluna].round(2)

    # Cria uma coluna com o valor total de obesidade por cidade e taxa de obesidade
    
    df_ano['obesidade_total'] = df_ano['Obesidade Grau I'] + df_ano['Obesidade Grau II'] + df_ano['Obesidade Grau III']

    df_ano['obesidade_total_%'] = ((df_ano['obesidade_total'] / df_ano['Total']) * 100).round(2)

    lista_dfs.append(df_ano)

df_sisvan_completo = pd.concat(lista_dfs, ignore_index=True)

In [ ]:
df_ano.columns

In [ ]:
df_ano.info()

In [ ]:
#lista_dfs[0] = Dados de 2020
#lista_dfs[1] = Dados de 2021
#lista_dfs[2] = Dados de 2022
#lista_dfs[3] = Dados de 2023
#lista_dfs[4] = Dados de 2024
#lista_dfs[5] = Dados de 2025

lista_dfs[0].head()

- Leitura do IDHM

In [7]:
caminho_idhm_mg = '5 - IDHM/IDHM - Minas Gerais.xlsx'
df_idhm_mg = pd.read_excel(caminho_idhm_mg)

# Tratamento e normalização dos nomes dos municípios
df_idhm_mg['municipio_norm'] = df_idhm_mg['Municípios'].apply(normalizar_texto)

# Mudança do tipo de dados da coluna IDHM para numérico
df_idhm_mg['idhm_2010'] = pd.to_numeric(df_idhm_mg['Índice de Desenvolvimento Humano Municipal (IDHM) 2010'], errors='coerce')

# Categorização nas faixas do PNUD
faixas = [0.0, 0.499, 0.599, 0.699, 0.799, 1.0]
rotulos = ['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']

df_idhm_mg['faixa_idhm'] = pd.cut(df_idhm_mg['idhm_2010'], bins=faixas, labels=rotulos)



In [ ]:
df_idhm_mg.head()

In [ ]:


linha_763 = df_idhm_mg.iloc[763]
print(linha_763)

- Escolhas das colunas que serão utilizadas no cruzamentos dos dados entre SISVAN e IDHM

In [10]:
colunas_desejadas = ['ano', 'UF', 'Código IBGE','municipio_norm', 'Baixo peso', 'Adequado', 'Sobrepeso', 'Obesidade Grau I', 'Obesidade Grau II', 'Obesidade Grau III', 'Total', 'obesidade_total', 'obesidade_total_%', 'Obesidade Grau I %', 'Obesidade Grau II %', 'Obesidade Grau III %',]

df_sisvan_limpo = df_sisvan_completo[colunas_desejadas].copy()


In [ ]:
total_municipios = df_sisvan_limpo['municipio_norm'].nunique()
total_municipios

In [12]:
# Substituições para ficar com o nome correto
substituicoes = {
    'DONA EUZEBIA': 'DONA EUSEBIA',
    'BARAO DO MONTE ALTO': 'BARAO DE MONTE ALTO',
    'SAO TOME DAS LETRAS': 'SAO THOME DAS LETRAS'
}

df_idhm_mg['municipio_norm'] = df_idhm_mg['municipio_norm'].replace(substituicoes)

- Cruzamento dos dados SISVAN e IDHM

In [13]:
df_mg_consolidado = pd.merge(
    df_sisvan_limpo,
    df_idhm_mg[['municipio_norm', 'idhm_2010', 'faixa_idhm']],
    on='municipio_norm',
    how = 'inner'
)

In [ ]:
# Municípios que estão no SISVAN mas não entraram no df_mg_consolidado
faltantes = set(df_sisvan_limpo['municipio_norm']) - set(df_mg_consolidado['municipio_norm'])

print("Municípios descartados no cruzamento:")
print(faltantes)

In [ ]:
total_municipios = df_mg_consolidado['municipio_norm'].nunique()
total_municipios

In [ ]:
df_mg_consolidado.head()

- Quantidade de municípios em cada uma das faixas de IDHM (como o IDHM é de 2010 a faixa é igual para todos os anos)

In [ ]:
contagem_df = df_mg_consolidado.groupby(['faixa_idhm'])['municipio_norm'].nunique().reset_index()

contagem_df.columns = ['Faixa de IDHM', 'Qtd Municípios']

contagem_df

*  Contagem de municípios com obesidade_total igual a 0 ou Nula por ano

In [ ]:
zerados_por_ano = df_mg_consolidado[(df_mg_consolidado['obesidade_total'] == 0) | (df_mg_consolidado['obesidade_total'].isna())].groupby('ano')['municipio_norm'].count()

print("Quantidade de municípios com obesidade zerada/nula por ano:")
print(zerados_por_ano)

- A cidade de Padre Carvalho no ano de 2021 não apresentou pessoas com obesidade

In [ ]:
# Lista os municípios com obesidade zerada em 2020
municipios_zerados_2021 = df_mg_consolidado[(df_mg_consolidado['ano'] == 2021) & ((df_mg_consolidado['obesidade_total'] == 0) | (df_mg_consolidado['obesidade_total'].isna()))
][['municipio_norm', 'Total', 'obesidade_total']]

municipios_zerados_2021

- Análise exploratória agregada de Minas Gerais no período 2020-2025

In [ ]:
# Agrupa por Faixa de IDHM e calcula a média e mediana da taxa de obesidade
comparacao_idhm = df_mg_consolidado.groupby('faixa_idhm').agg(
     qtd_municipios=('municipio_norm', 'nunique'),
     qtd_observacoes=('municipio_norm', 'count'),
     media_obesidade=('obesidade_total_%', 'mean'),
     mediana_obesidade=('obesidade_total_%', 'median')
).round(2).reset_index()


# Exibe o resultado formatado
comparacao_idhm.rename(columns={
    'faixa_idhm': 'Faixa do IDHM',
    'qtd_municipios': 'Número de Municípios',
    'qtd_observacoes': 'Número de observações (2020-2025)',
    'media_obesidade': 'Média da Taxa de Obesidade (%)',
    'mediana_obesidade': 'Mediana da Taxa de Obesidade (%)'
})

- Tabela de cruzamento **Média** por Ano e a Faixa de IDHM

In [ ]:
evolucao_anual = df_mg_consolidado.groupby(['faixa_idhm', 'ano'])['obesidade_total_%'].mean().unstack()

# Arredonda para 2 casas decimais
evolucao_anual.round(2)

- Tabela de cruzamento **Mediana** por Ano e a Faixa de IDHM

In [ ]:
evolucao_anual_mediana = df_mg_consolidado.groupby(['faixa_idhm', 'ano'])['obesidade_total_%'].median().unstack()

# Arredonda para 2 casas decimais
evolucao_anual_mediana.round(2)

* **Gráfico de Dispersão IDHM x Taxa de Obesidade**

In [ ]:
# Seleção e remoção de valores nulos das colunas
df_valido = df_mg_consolidado[['idhm_2010', 'obesidade_total_%']].dropna()

x = df_valido['idhm_2010']
y = df_valido['obesidade_total_%']

#  Cálculo da linha de tendência 
m, b = np.polyfit(x, y, 1)

# Construção do gráfico com Matplotlib
plt.figure(figsize=(7, 4))

# Ponto de dispersão
plt.scatter(x, y, alpha=0.4, color="#1b8bdb", label='Municípios')

# Linha de tendência
plt.plot(x, m * x + b, color='red', linewidth=2, label=f'Tendência')

# Plotagem do gráfico
plt.title('Dispersão: IDHM (2010) vs. Taxa de Obesidade (%)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('IDHM (2010)', fontsize=11)
plt.ylabel('Taxa de Obesidade (%)', fontsize=11)
plt.grid(True, linestyle='--', alpha=0.6)

plt.legend(
    title='Legenda',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0,
)

plt.tight_layout()
plt.show()

- **Verificação da relação dos Graus de Obesidade e do IDHM 2020 - 2025**

In [ ]:
grupo_grau_de_obesidade = df_mg_consolidado.groupby('faixa_idhm')[['Obesidade Grau I %', 'Obesidade Grau II %','Obesidade Grau III %']].mean().round(2)

grupo_grau_de_obesidade.style.format("{:.2f}").set_caption("Evolução Média dos Graus de Obesidade (%) por Faixa de IDHM (2020-2025)").set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold'), ('font-size', '11pt')]}])

- Evolução anual **Obesidade Grau I** em relação as faixas de IDHM

In [ ]:
evolucao__anual_GrauI = df_mg_consolidado.groupby(['faixa_idhm', 'ano'])['Obesidade Grau I %'].mean().unstack()

evolucao__anual_GrauI.style.format("{:.2f}").set_caption("Evolução da Obesidade Grau I (%) por Faixa de IDHM (2020–2025)").set_table_styles([{'selector': 'caption','props':[('font-weight', 'bold'), ('font-size', '11pt')]}])

- Evolução anual **Obesidade Grau II** em relação as faixas de IDHM

In [ ]:
evolucao__anual_GrauII = df_mg_consolidado.groupby(['faixa_idhm', 'ano'])['Obesidade Grau II %'].mean().unstack()

evolucao__anual_GrauII.style.format("{:.2f}").set_caption("Evolução da Obesidade Grau II (%) por Faixa de IDHM (2020–2025)").set_table_styles([
    {'selector': 'caption','props':[('font-weight', 'bold'), ('font-size', '11pt')]}])

- Evolução anual **Obesidade Grau III** em relação as faixas de IDHM

In [ ]:
evolucao__anual_GrauIII = df_mg_consolidado.groupby(['faixa_idhm', 'ano'])['Obesidade Grau III %'].mean().unstack()

evolucao__anual_GrauIII.style.format("{:.2f}").set_caption("Evolução da Obesidade Grau III (%) por Faixa de IDHM (2020–2025)").set_table_styles([{'selector': 'caption','props':[('font-weight', 'bold'), ('font-size', '11pt')]}])

* Prevalência agregada de obesidade entre os indivíduos avaliados no SISVAN

In [ ]:
totais_por_faixa = df_mg_consolidado.groupby(['ano', 'faixa_idhm'])[['obesidade_total', 'Total']].sum()

totais_por_faixa['prevalencia %'] = (totais_por_faixa['obesidade_total'] / totais_por_faixa['Total'] * 100).round(2)

totais_por_faixa

In [ ]:
df_pivot = totais_por_faixa['prevalencia %'].unstack()

plt.figure(figsize=(9, 4))

faixas = ['Baixo', 'Médio', 'Alto', 'Muito Alto']

for faixa in faixas:
    if faixa in df_pivot.columns:
        plt.plot(
            df_pivot.index,
            df_pivot[faixa],
            marker='o',
            linewidth=2.5,
            label=f'IDHM {faixa}',
        )

        for ano in df_pivot.index:
            valor = df_pivot.loc[ano, faixa]
            plt.annotate(
                f'{valor:.2f}%',
                (ano, valor),
                textcoords='offset points',
                xytext=(0, 7),
                ha='center',
                fontsize=8.5,
                fontweight='bold',
            )

plt.title(
    'Prevalência agregada de obesidade entre os indivíduos avaliados no SISVAN',
    fontsize=11,
    fontweight='bold',
    pad=10,
)
plt.xlabel('Ano', fontsize=11)
plt.ylabel('Prevalência de Obesidade (%)', fontsize=11)

plt.ylim(18, 48)
plt.xticks(df_pivot.index)
plt.grid(True, linestyle='--', alpha=0.5)

plt.legend(
    title='Faixa IDHM',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    borderaxespad=0,
)

plt.tight_layout()
plt.show()

* **Nomes dos municípios em cada uma das faixas de IDHM**

In [ ]:
cidades_idhm_muito_alto = df_mg_consolidado[(df_mg_consolidado['faixa_idhm'] == 'Muito Alto') & (df_mg_consolidado ['ano'] == 2020)][['municipio_norm', 'faixa_idhm', 'idhm_2010']]

cidades_idhm_muito_alto



In [ ]:
cidades_idhm_baixo = df_mg_consolidado[(df_mg_consolidado['faixa_idhm'] == 'Baixo') & (df_mg_consolidado['ano'] == 2020)][['municipio_norm', 'faixa_idhm', 'idhm_2010']]

'''print(cidades_idhm_baixo['municipio_norm'].tolist())

with pd.option_context('display.max_rows', None):
    display(cidades_idhm_baixo)'''

cidades_idhm_baixo

In [ ]:
cidades_idhm_medio = df_mg_consolidado[(df_mg_consolidado['faixa_idhm'] == 'Médio') & (df_mg_consolidado['ano'] == 2020)][['municipio_norm', 'faixa_idhm', 'idhm_2010']]

'''print(cidades_idhm_medio['municipio_norm'].tolist())

with pd.option_context('display.max_rows', None):
    display(cidades_idhm_medio)'''

cidades_idhm_medio

In [ ]:
cidades_idhm_alto = df_mg_consolidado[(df_mg_consolidado['faixa_idhm'] == 'Alto') & (df_mg_consolidado['ano'] == 2020)][['municipio_norm', 'faixa_idhm', 'idhm_2010']]

'''print(cidades_idhm_alto['municipio_norm'].tolist())

with pd.option_context('display.max_rows', None):
    display(cidades_idhm_alto)'''

cidades_idhm_alto